In [4]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Praveen Kumar N\AppData\Local\Temp\ipykernel_22276\329546744.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
c:\AYUSH N P\Agentic-AI course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def process_pdf(pdf_dir):

    all_doc=[]
    pdf_dir = Path(pdf_dir)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process") 
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file
                doc.metadata['file_type'] = 'pdf'
                
            all_doc.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
        except Exception as e:
            print(f"  ✗ Error loading {pdf_file.name}: {e}")
    print(f"\nTotal documents loaded: {len(all_doc)}")
    return all_doc

all_pdf_documents = process_pdf("../data/pdf_files")


Found 3 PDF files to process

Processing: 1706.03762v7.pdf
  ✓ Loaded 15 pages

Processing: 2506.18027v3.pdf
  ✓ Loaded 14 pages

Processing: iso27001.pdf
  ✓ Loaded 26 pages

Total documents loaded: 55


In [6]:
from pprint import pprint
pprint(all_pdf_documents)

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'source': '..\\data\\pdf_files\\1706.03762v7.pdf', 'file_path': '..\\data\\pdf_files\\1706.03762v7.pdf', 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'modDate': 'D:20240410211143Z', 'creationDate': 'D:20240410211143Z', 'page': 0, 'source_file': WindowsPath('../data/pdf_files/1706.03762v7.pdf'), 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\

In [7]:
def split_docs(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [8]:
chunks = split_docs(all_pdf_documents)
chunks

Split 55 documents into 187 chunks

Example chunk:
Content: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'source': '..\\data\\pdf_files\\1706.03762v7.pdf', 'file_path': '..\\data\\pdf_files\\1706.03762v7.pdf', 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'modDate': 'D:20240410211143Z', 'creationDate': 'D:20240410211143Z', 'page': 0, 'source_file': WindowsPath('../data/pdf_files/1706.03762v7.pdf'), 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'source': '..\\data\\pdf_files\\1706.03762v7.pdf', 'file_path': '..\\data\\pdf_files\\1706.03762v7.pdf', 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'modDate': 'D:20240410211143Z', 'creationDate': 'D:20240410211143Z', 'page': 0, 'source_file': WindowsPath('../data/pdf_files/1706.03762v7.pdf'), 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\

### embedding And vectorStoreDB

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [11]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self,model_name:str = "all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded model: {self.model_name}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
    def generate_embedding(self,text:List[str])->np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded. Call _load_model() first.")
        
        embeddings = self.model.encode(text,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
embedding_manager = EmbeddingManager()
embedding_manager



Loaded model: all-MiniLM-L6-v2


### vectorStore

In [12]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    def __init__(self,collection_name="pdf_documents",persistent_dir="../data/vector_store"):
         """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
         self.collection_name = collection_name
         self.persistent_dir = persistent_dir
         self.client=None
         self.collection=None
         self._initialize_store()

    def _initialize_store(self):
         
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persistent_dir, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persistent_dir)

            self.collection = self.client.get_or_create_collection(name=self.collection_name,
                                                                     metadata={"description": "PDF document embeddings"})
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
                print(f"Error initializing vector store: {e}")
                raise

    def add_documents(self,documents:List[Any],embeddings:np.ndarray):

        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: numpy array of embeddings corresponding to the documents"""  
        
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to vector store...")

        ids=[]
        metadatas=[]
        doc_text=[]
        embedding_list=[]

        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)

            from pathlib import Path

            for key, value in metadata.items():
                if isinstance(value, Path):
                    metadata[key] = str(value)

            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            doc_text.append(doc.page_content)

            embedding_list.append(embedding.tolist())

        try:
            self.collection.add(
                  ids=ids,
                  metadatas=metadatas,
                  documents=doc_text,
                  embeddings=embedding_list
             )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [13]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embedding(texts)

vectorstore.add_documents(chunks, embeddings)

Batches: 100%|██████████| 6/6 [00:07<00:00,  1.30s/it]


Generated embeddings with shape: (187, 384)
Adding 187 documents to vector store...
Successfully added 187 documents to vector store
Total documents in collection: 187


In [ ]:
class RagRetriever:
    """Retrieves relevant documents from the vector store based on a query"""

    def __init__(self, vectorstore:VectorStore, embedding_manager:EmbeddingManager):
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager


    def retrieve(self,query:str,top_k:int=5,score_threshold:float=0.0)->List[Dict[str,Any]]:
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            print(len(results["documents"][0]))
            print(len(results["ids"][0]))
            print(len(results["distances"][0]))
            retrieved_docs=[]

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i , (doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    similarity_score =1-distance
                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            "id":doc_id,
                            "document":document,
                            "metadata":metadata,
                            "similarity_score":similarity_score,
                            "distance":distance,
                            "rank":i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents above the score threshold")
            else:
                print("No documents retrieved from the vector store.")

            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
    
rag_retriever = RagRetriever(vectorstore,embedding_manager)



In [15]:
rag_retriever

In [16]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 27.55it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents above the score threshold


[{'id': 'doc_b4f6d90d_12',
  'document': '3.2\nAttention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3',
  'metadata': {'creationdate': '2024-04-10T21:11:43+00:00',
   'title': '',
   'author': '',
   'total_pages': 15,
   'producer': 'pdfTeX-1.40.25',
   'file_type': 'pdf',
   'source': '..\\data\\pdf_files\\1706.03762v7.pdf',
   'trapped': '',
   'doc_index': 12,
   'source_file': '..\\data\\pdf_files\\1706.03762v7.pdf',
   'file_path': '..\\data\\pdf_files\\1706.03762v7.pdf',
   'creationDate': 'D:20240410211143Z',
   'keywords': '',
   'format': 'PDF 1.5',
   'subject': '',
   'page': 2,
   'modDate': 'D:20240410211143Z',
   'creator': 'LaTeX with hyperref',
   'content_length': 216,
   'moddate': '2024-04-10T21:11:43+00:00'},
  'similarity_score': 0.1399548053741455,
  'distance': 0.8600451946258545,
  'rank': 1}]

In [17]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")


Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.26it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents above the score threshold


[]

### pipeline db->llm

In [18]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()


True

In [19]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.1,
    max_tokens=1024
)



In [25]:
from langchain_core.messages import HumanMessage


def rag_simple(llm,retriver,query,top_k=3):

    results = retriver.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['document'] for doc in results]) if results else " "
    if not context.strip():
        return "No relevant documents found."

    prompt = f""" Use the following context to answer the question consisely. Context :{context} Question: {query} If the context does not contain the answer, respond with 'I don't know'."""
    print(prompt)
    response = llm.invoke(
        [HumanMessage(content=prompt)]
    )
    return response.content.strip()

In [26]:
answer = rag_simple(llm,rag_retriever,"What is attention is all you need")
print(answer)

Retrieving documents for query: 'What is attention is all you need'
Top K: 3, Score threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.24it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents above the score threshold
 Use the following context to answer the question consisely. Context :3.2
Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3 Question: What is attention is all you need If the context does not contain the answer, respond with 'I don't know'.


I don't know.


In [27]:
print(vectorstore.collection.count())

187
